# 交易分类回溯（Traceback）与全量批量分类（仓库根单文件入口）

对根目录 [classification_traceback.py](classification_traceback.py) 的 notebook 封装
（合并自 backfill/ 独立子项目；镜像 model_development / backfill/main_traceback.ipynb 的
RUN_MODE + TEST_MODE 组织方式）。模型接入 = **直连仓库引擎代码**（`ClassificationOrchestrator(
load_pipeline_config(), load_category_owners())`），无 pkl 注册层——改引擎/规则/配置后
无需重建任何模型文件，run_meta.json（v2）自动记录 configs 与规则资源指纹。

三种运行模式（`RUN_MODE` 切换）：

- **batch 模式**：单份交易 CSV 全量跑一遍分类流水线（943 平台契约），
  输出**四个数据文件夹**（transactions / income_summary / liability_summary / category_summary），
  大文件按 `CHUNK_ROWS` 自动分块。
- **classification 模式**：按 application_id 回溯——从语料提取每个申请的全部交易 → 重跑流水线 →
  输出四个数据文件夹（含行级 transactions，按行分块）。**两种输入二选一**：
  - 填 `TXN_INPUT`（单份大交易 CSV，如 spark 导出或 illion 月语料）→ 自动按 app 切分 +
    自动清洗脏格式 + 逐 app 多进程（同 CLI `--input` 语义）；
  - 否则用 `SAMPLE_PATH`（样本清单）+ `TXN_DIR`（已切分的语料目录，如某次
    classification 运行产出的 `.corpus_single`）。
- **summary 模式**：app 级输出（三个 summary 文件夹，不写行级明细）；
  需要 `SAMPLE_PATH` + `TXN_DIR`（填 classification 运行产出的 samples_auto.csv + .corpus_single 即可）。

classification / summary 支持多进程（multiprocessing.Pool）、断点续跑（.progress.txt 记录已完成 app）、
错误快照（失败 app 记入 error_detail.csv，含输入 JSON）；填已有 `OUT_DIR` = 对旧目录断点续跑，
`REPLACE = True` 清块清进度全量重跑。

> ⚠️ **Windows + Jupyter 下 multiprocessing.Pool(spawn) 不稳定**：本机（Windows）notebook 内
> 建议 `N_WORKERS = 1`，要真多进程请用脚本方式
> `python classification_traceback.py --mode traceback --input ... --workers N`；
> **线上 Linux 没有此问题**，`N_WORKERS = None`（自动 = cpu-2）或填数字即可。
> 本机内存经验：每 worker ~1.2GB（initial 引擎 automaton 大），`N_WORKERS * 1.2GB + 6GB`
> 需小于可用内存（本机安全上限约 6-8）。
>
> 单申请 JSON 排查（`--mode app`，带 engineClaims 认领层）适合走 CLI：
> `python classification_traceback.py --mode app --input app.json --output out.json`


In [ ]:
# =====================================================
# 定位仓库根并引入根目录单文件（kernel cwd 不假设；本文件在仓库根时即命中）
# =====================================================
import os
import sys
from pathlib import Path

ROOT = Path(os.getcwd()).resolve()
if not (ROOT / "classification_traceback.py").exists():
    for _parent in ROOT.parents[:4]:
        if (_parent / "classification_traceback.py").exists():
            ROOT = _parent
            break
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("ROOT =", ROOT)

In [ ]:
# =====================================================
# ⭐ 模式与路径切换（只改这里）
# =====================================================
import classification_traceback  # 根目录单文件：batch/traceback/summary/app 全入口

RUN_MODE = "classification"  # "batch" | "classification" | "summary"
TEST_MODE = True             # True = 跑仓库根自带样本；False = 生产路径（填下方真实路径）

if TEST_MODE:
    # 仓库根自带样本（git 跟踪）：sample_35_apps.csv = 35 apps / 44,241 笔（~10s/次）
    BATCH_INPUT = ROOT / "sample_35_apps.csv"      # batch 模式输入
    TXN_INPUT = ROOT / "sample_35_apps.csv"        # classification 模式输入（单文件自动切 app）
    SAMPLE_PATH = None       # classification/summary 样本清单；TXN_INPUT 模式保持 None
    TXN_DIR = None           # classification/summary 语料目录；TXN_INPUT 模式保持 None
    OUT_DIR = ROOT / "output" / "notebook_traceback"  # 落 output/（gitignored）
else:
    BATCH_INPUT = Path(r"<单份交易 CSV，batch 模式必填>")  # 全量批量输入（batch 整帧进内存，大文件跑不动）
    TXN_INPUT = Path(r"<单份交易 CSV，classification 可选>")
                             # 填 TXN_INPUT = 自动切 app + 自动清洗 + 逐 app 多进程（同 CLI --input）
    SAMPLE_PATH = Path(r"<样本清单 CSV，每行至少 application_id 列>")  # 或 None
    TXN_DIR = Path(r"<语料目录>")                     # 或 None
    OUT_DIR = None           # None = 新建 output/traceback_{时间戳}；填已有目录 = 断点续跑

CHUNK_ROWS = 500_000        # 每个数据文件夹按多少行分一块
N_WORKERS = 1               # 1 = 纯顺序执行（Windows + Jupyter 稳妥）；Linux 可 None=自动 cpu-2
REPLACE = False             # True = 清块 + 清进度，重新全量跑

In [ ]:
# =====================================================
# 执行（通常不用改）
# =====================================================
if RUN_MODE == "batch":
    out_dir = classification_traceback.run_batch_pipeline(
        input_csv=BATCH_INPUT, out_dir=OUT_DIR, chunk_rows=CHUNK_ROWS,
    )
elif RUN_MODE == "classification":
    if TXN_INPUT is not None:
        out_dir = classification_traceback.run_single_file_pipeline(
            input_csv=TXN_INPUT, out_dir=OUT_DIR, chunk_rows=CHUNK_ROWS,
            n_workers=N_WORKERS, replace=REPLACE,
        )
    else:
        out_dir = classification_traceback.run_classification_pipeline(
            sample_path=SAMPLE_PATH, txn_dir=TXN_DIR, out_dir=OUT_DIR,
            chunk_rows=CHUNK_ROWS, n_workers=N_WORKERS, replace=REPLACE,
        )
elif RUN_MODE == "summary":
    out_dir = classification_traceback.run_summary_pipeline(
        sample_path=SAMPLE_PATH, txn_dir=TXN_DIR, out_dir=OUT_DIR,
        chunk_rows=CHUNK_ROWS, n_workers=N_WORKERS, replace=REPLACE,
    )
else:
    raise SystemExit(f"未知 RUN_MODE: {RUN_MODE!r}（batch | classification | summary）")

print(f"✅ 输出目录：{out_dir}")

## 结果预览

下面几个 cell 只读上面 `out_dir` 的产物，不改任何文件。

In [ ]:
# ── 输出总览：各数据文件夹的行数 / 块数 ──
import glob
import pandas as pd

def _load_parts(out_dir, name):
    """读数据文件夹的块（out_dir/{name}.csv/{name}_*.csv），拼接。"""
    paths = sorted(glob.glob(str(out_dir / f"{name}.csv" / f"{name}_*.csv")))
    return [pd.read_csv(p) for p in paths] if paths else []

for name in ("transactions", "income_summary", "liability_summary", "category_summary"):
    parts = _load_parts(out_dir, name)
    total = sum(len(p) for p in parts)
    print(f"{name}.csv/: {len(parts)} 块, {total} 行")

In [ ]:
# ── 抽查：unclassified 交易与赢家引擎分布 ──
txn_parts = _load_parts(out_dir, "transactions")
if txn_parts:
    txns = pd.concat(txn_parts, ignore_index=True)
    print("交易总数：", len(txns))
    print("unclassified：", int((txns["classification_status"] == "unclassified").sum()))
    print("赢家引擎分布：")
    print(txns["classification_engine"].value_counts())

In [ ]:
# ── 分类详情查询：某 app 的某笔交易最终分成了什么 ──
# （engine_claims 认领层不在 CSV 产物内；"为什么最终是 X"如需排查，可跑 CLI --mode app）
TARGET_APP = None          # ← 换成 application_id 字符串；None = 先列出样本里有哪些 app
TARGET_TXN = None          # 留 None 看该 app 全部交易

txn_parts = _load_parts(out_dir, "transactions")
if txn_parts:
    txns = pd.concat(txn_parts, ignore_index=True)
    txns["application_id"] = txns["application_id"].astype(str)
    if TARGET_APP is None:
        print("样本中的 app（前 10）：")
        print(txns["application_id"].value_counts().head(10).to_string())
    else:
        sub = txns[txns["application_id"] == str(TARGET_APP)]
        if TARGET_TXN is not None:
            sub = sub[sub["transaction_id"].astype(str) == str(TARGET_TXN)]
        print(f"{TARGET_APP} 共 {len(sub)} 笔")
        sub[["transaction_id", "bscat", "counterparty", "classification_engine",
             "classification_rule_id", "classification_reason"]].head(20)

In [ ]:
# ── 三个汇总预览（按键：application_id / bank_account_id / bscat / stream_id） ──
from IPython.display import display

for name in ("income_summary", "liability_summary", "category_summary"):
    parts = _load_parts(out_dir, name)
    if parts:
        df = pd.concat(parts, ignore_index=True)
        print(f"═══ {name}（{len(df)} 行）═══")
        display(df.head(5))
    else:
        print(f"（无 {name} 产物）")